Project part 1:
Shirel Amar 207065103 & Sarah Derhy 340889435
https://github.com/SarahDerhy/Data-analyse-advanced-in-Python

In [2]:
import requests
from bs4 import BeautifulSoup
import json
import time
import pandas as pd
from pathlib import Path
import re
import io
from tqdm import tqdm

In [ ]:
#Importing our tabs directly from the IMDB website

In [ ]:
BASE_URL = "https://datasets.imdbws.com/"

folders = {
    "basics":     "title.basics.tsv.gz",
    "ratings":    "title.ratings.tsv.gz",
    "principals": "title.principals.tsv.gz",
    "names":      "name.basics.tsv.gz"
}

def download_file(folder_name):
    url = BASE_URL + folder_name
    response = requests.get(url, stream=True)
    return io.BytesIO(response.content)

basics_file     = download_file(folders["basics"])
ratings_file    = download_file(folders["ratings"])
principals_file = download_file(folders["principals"])
names_file      = download_file(folders["names"])

In [ ]:
#The data is too big, so we cut it into chunks and import it chunk by chunk

In [ ]:
basics_chunks = []
for chunk in pd.read_csv(basics_file,
    sep="\t",
    na_values="\\N",
    on_bad_lines="skip",
    compression="gzip",
    low_memory=False,
    chunksize=100_000):
    
    chunk = chunk[(chunk["titleType"] == "movie") & 
                  (chunk["primaryTitle"].str.startswith(("K", "k", "N", "n"), na=False))]
    basics_chunks.append(chunk)

basics = pd.concat(basics_chunks, ignore_index=True)

In [ ]:
basics.head()

In [ ]:
basics_tconst = set(basics["tconst"]) 
rating_chunks = []
for chunk in pd.read_csv(ratings_file,
    sep="\t",
    na_values="\\N",
    on_bad_lines="skip",
    compression="gzip",
    low_memory=False,
    chunksize=100_000):
    
    chunk = chunk[chunk["tconst"].isin(basics_tconst)]
    rating_chunks.append(chunk)

ratings = pd.concat(rating_chunks, ignore_index=True)


In [ ]:
ratings.head()

In [ ]:
principals_chunks = []
for chunk in pd.read_csv(principals_file,
    sep="\t",
    na_values="\\N",
    on_bad_lines="skip",
    compression="gzip",
    low_memory=False,
    chunksize=100_000):
    
    chunk = chunk[chunk["tconst"].isin(basics_tconst)]
    principals_chunks.append(chunk)

principals = pd.concat(principals_chunks, ignore_index=True)


In [ ]:
principals.head()

In [ ]:
principals_nconst = set(principals["nconst"])

names_chunks = []
for chunk in pd.read_csv(names_file,
    sep="\t",
    na_values="\\N",
    on_bad_lines="skip",
    compression="gzip",
    low_memory=False,
    chunksize=100_000):
    
    chunk = chunk[chunk["nconst"].isin(principals_nconst)]
    names_chunks.append(chunk)

names = pd.concat(names_chunks, ignore_index=True)

In [ ]:
names.head()

In [ ]:
#Tabs are loaded, we drop the column we need.

In [ ]:
basics= basics.drop(columns=["originalTitle", "isAdult","endYear"])
principals= principals.drop(columns=["ordering","job","characters"])
names= names[["nconst"]]

In [ ]:
#We keep in "principals" only "actor" and "actress" .

In [ ]:
principals= principals[principals["category"].isin(["actor", "actress"])]

In [ ]:
#We merge our tabs together.

In [ ]:
principals_names = principals.merge(names, on="nconst", how="left")

In [ ]:
df=basics.merge(ratings, on="tconst", how="left")

In [ ]:
df= df.merge(principals_names, on="tconst", how="left")

In [ ]:
#Adding more filter on our data.

In [ ]:
df= df[df["startYear"] <= 2024]
df["runtimeMinutes"]= pd.to_numeric(df["runtimeMinutes"], errors="coerce")
df= df[(df["runtimeMinutes"] >= 60) & (df["runtimeMinutes"] <= 300)]

In [ ]:
df.sample()

In [ ]:
#Creating a new column: "lead_actors_ids".

In [ ]:
top5 = (df[df["category"].isin(["actor", "actress"])]
               .groupby("tconst")["nconst"]
               .apply(lambda x: list(x)[:5])
               .reset_index()
               .rename(columns={"nconst": "lead_actors_ids"}))
df = df.merge(top5, on="tconst", how="left")

In [ ]:
df.sample()

In [ ]:
df['tconst'].nunique()

We reduced our dataset from 21,265 to approximately 6000 movies by keeping only movies with at least 100 votes, as they are more likely to have information available on Wikipedia.

In [ ]:
df = df[df["numVotes"] >= 100]

In [ ]:
df['tconst'].nunique()

In [ ]:
#For now we have multiple rows for the same movie. 
#Since the "lead_actor_ids" column already groups the actors, we can remove duplicates and keep one row per movie.

In [ ]:
df_unique = df.drop_duplicates(subset="tconst")

In [ ]:
#We start from now the websrapping from Wikipedia to complete our data set.

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0 Safari/537.36"
}

In [ ]:
#We have a clean function to have our final results written in a clean way.

In [ ]:
def clean(value):
    if value:
        value = value.replace("\xa0", " ")
        value = re.sub(r'\[.*?\]', '', value)
        return value.strip()
    return None



In [ ]:
#normalize function: normalize title by removing symbols and splitting into set of lowercase words for comparison
#get_movie_info function: Search on Wikipedia by title and year, extract language, country, budget and box office from the infobox and the plot from the page, return all as a dictionary or None if not found.

In [ ]:
def normalize(text):
    return set(re.sub(r"[^\w\s]", "", text.lower()).split())

def get_movie_info(primaryTitle, year):
    empty = {
        "language": None,
        "country": None,
        "plot": None,
        "budget": None,
        "boxOffice": None
    }

    try:
        search_url = "https://en.wikipedia.org/w/api.php"
        search_params = {
            "action": "query",
            "list": "search",
            "srsearch": f"{primaryTitle} {year} film".strip() if year else f"{primaryTitle} film",
            "srnamespace": "0",
            "srlang": "en",
            "format": "json"
        }

        response = requests.get(search_url, params=search_params, headers=headers, timeout=10)

        if response.status_code == 429:
            wait = int(response.headers.get("Retry-After", 60))
            print(f"Rate limited. Wait {wait}s...")
            time.sleep(wait)
            return get_movie_info(primaryTitle, year)

        if response.status_code != 200:
            return empty

        data = response.json()

        if not data["query"]["search"]:
            return empty

        title_words = normalize(primaryTitle)
        best_with_year = None
        best_without_year = None

        for result in data["query"]["search"]:
            page_words = normalize(result["title"])
            page_title_lower = result["title"].lower()

            title_match = title_words.issubset(page_words)
            year_match = year and str(year) in page_title_lower

            if title_match and year_match and best_with_year is None:
                best_with_year = result["title"]

            if title_match and not year_match and best_without_year is None:
                best_without_year = result["title"]

        if best_with_year:
            page_title = best_with_year
        elif best_without_year:
            page_title = best_without_year
        else:
            return empty

        wiki_url = "https://en.wikipedia.org/wiki/" + page_title.replace(" ", "_")

        page_response = requests.get(wiki_url, headers=headers, timeout=10)

        if page_response.status_code == 429:
            wait = int(page_response.headers.get("Retry-After", 60))
            print(f"Rate limited. Wait {wait}s...")
            time.sleep(wait)
            return get_movie_info(primaryTitle, year)

        if page_response.status_code != 200:
            return empty

        soup = BeautifulSoup(page_response.content, "html.parser")

        infobox = soup.find("table", {"class": "infobox"})
        language = None
        country = None
        budget = None
        box_office = None

        if infobox:
            for row in infobox.find_all("tr"):
                header = row.find("th")
                value = row.find("td")

                if header and value:
                    header_text = header.get_text().strip().lower()
                    value_text = value.get_text().strip()

                    if "language" in header_text:
                        language = value_text
                    elif "countr" in header_text:     #Countr to get country AND countries
                        country = value_text
                    elif "budget" in header_text:
                        budget = value_text
                    elif "box office" in header_text or "boxoffice" in header_text:
                        box_office = value_text

        plot = None
        plot_header = soup.find(
            lambda tag: tag.name in ["h2", "h3"] and "plot" in tag.get_text().lower()
        )

        if plot_header:
            next_p = plot_header.find_next("p")

            if next_p:
                plot = next_p.get_text().strip()

        time.sleep(1.5)

        return {
            "language": clean(language),
            "country": clean(country),
            "plot": plot,
            "budget": clean(budget),
            "boxOffice": clean(box_office)
        }

    except Exception as e:
        print(f"Error for '{primaryTitle}': {e}")
        return empty

In [ ]:
#We run the get_movies_info function by chunk to not reach the rate limite from Wikipedia.

In [ ]:
chunk_size = 50
chunks = [df_unique[i:i+chunk_size] for i in range(0, len(df_unique), chunk_size)]

all_results = []
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1}/{len(chunks)} ---")
    
    for _, row in tqdm(chunk.iterrows(), total=len(chunk)):
        info = get_movie_info(row["primaryTitle"], row["startYear"])
        info["tconst"] = row["tconst"]
        all_results.append(info)
    
    pd.DataFrame(all_results).to_csv("wikipedia_results_checkpoint.csv", index=False)
    print(f"Chunked saved nº{i+1} ({len(all_results)} films processed)")
    
    if i < len(chunks) - 1:
        print(f"10sec break before the next chunk.")
        time.sleep(10)

wikipedia_df = pd.DataFrame(all_results)
final_df = df_unique.merge(wikipedia_df, on="tconst", how="left")
final_df.to_csv("final_df.csv", index=False)
print("Scraping complete.")

In [ ]:
final_df.sample()

In [ ]:
#Checking how many null values we have in each column.

In [ ]:
final_df.isna().sum()